# Speedup decision process with a Rules Engine 

To accelerate our claims decision process, we can setup pre-defined static checks that can be applied without requiring a human in the loop.

These rules will leverage our previous Deep Learning image analysis. 

Typically, minor damage can be automatically validated, but if the reported damage doesn't match what our AI model found, we'll flag the claim to involve additional human investigation (Checks on policy coverage, assessed severity, accident location and speed limit violations)

## Implementing Dynamic rules

Many system exists to build such rules. These can be natively implemented in Spark or using Spark Declarative Pipelines (check `dbdemos.install('declarative-pipeline-unit-test')` for an example of dynamic SDP rules)

In this simple example, we'll add our rules as SQL statement in a table table and then apply them over our dataset.

This will allow our business to easily add / edit rules by simply adding a SQL entry to our table.

* Rule definition inludes:
  * Unique Rule name/id
  * Definition of acceptable and not aceptable data - written as code that can be directly applied
  * Severity (HIGH, MEDIUM, LOW)
  * Is Active (True/False)

* Some common checks include
  * Claim date should be within coverage period
  * Reported Severity should match ML predicted severity
  * Accident Location as reported by telematics data should match the location as reported in claim
  * Speed limit as reported by telematics should be within speed limits of that region if there is a dispute on who was on the offense 


  <img width="800px" src="https://raw.githubusercontent.com/databricks-demos/dbdemos-resources/main/images/fsi/smart-claims/rule_engine.png" />


<!-- Collect usage data (view). Remove it to disable collection or disable tracker during installation. View README for more details.  -->
<img width="1px" src="https://ppxrzfxige.execute-api.us-west-2.amazonaws.com/v1/analytics?category=lakehouse&org_id=2162748966026566&notebook=%2F02-Data-Science-ML%2F02.3-Dynamic-Rule-Engine&demo_name=lakehouse-fsi-smart-claims&event=VIEW&path=%2F_dbdemos%2Flakehouse%2Flakehouse-fsi-smart-claims%2F02-Data-Science-ML%2F02.3-Dynamic-Rule-Engine&version=1">

In [0]:
%run ../_resources/00-setup

run done
USE CATALOG `main__build`
using catalog.database `main__build`.`dbdemos_fsi_smart_claims`


data already existing.


In [0]:
%sql
CREATE OR REPLACE TABLE claims_rules (
  rule_id BIGINT GENERATED ALWAYS AS IDENTITY,
  rule STRING, 
  check_name STRING,
  check_code STRING,
  check_severity STRING,
  is_active Boolean
);
ALTER TABLE claims_rules SET OWNER TO `account users`;

## Creating our Rules

### Invalid Policy Date

In [0]:
def insert_rule(rule, name, code, severity, is_active):
  spark.sql(f"INSERT INTO claims_rules(rule,check_name, check_code, check_severity,is_active) values('{rule}', '{name}', '{code}', '{severity}', {is_active})")

invalid_policy_date = '''
  CASE WHEN to_date(pol_eff_date, "yyyy-MM-dd") < to_date(claim_date) and to_date(pol_expiry_date, "yyyy-MM-dd") < to_date(claim_date) THEN "VALID" 
  ELSE "NOT VALID"  
  END
'''

insert_rule('invalid policy date', 'valid_date', invalid_policy_date, 'HIGH', True)

### Exceeds Policy Amount

In [0]:
exceeds_policy_amount = '''
CASE WHEN  sum_insured >= claim_amount_total 
    THEN "calim value in the range of premium"
    ELSE "claim value more than premium"
END 
'''
insert_rule('exceeds policy amount', 'valid_amount', exceeds_policy_amount, 'HIGH', True)

### Severity Mismatch

In [0]:
severity_mismatch = '''
CASE WHEN    damage_prediction.label="major" AND damage_prediction.score > 0.8 THEN  "Severity matches the report"
       WHEN  damage_prediction.label="minor" AND damage_prediction.score > 0.6 THEN  "Severity matches the report"
       WHEN  damage_prediction.label="ok" AND damage_prediction.score > 0.4 THEN  "Severity matches the report"
       ELSE "Severity does not match"
END 
'''

insert_rule('severity mismatch', 'reported_severity_check', severity_mismatch, 'HIGH', True)

### Exceeds Speed

In [0]:
exceeds_speed = '''
CASE WHEN  telematics_speed <= 45 and telematics_speed > 0 THEN  "Normal Speed"
       WHEN telematics_speed > 45 THEN  "High Speed"
       ELSE "Invalid speed"
END
'''
insert_rule('exceeds speed', 'speed_check', exceeds_speed, 'HIGH', True)

In [0]:
release_funds = '''
CASE WHEN  reported_severity_check="Severity matches the report" and valid_amount="calim value in the range of premium" and valid_date="VALID" then "release funds"
       ELSE "claim needs more investigation" 
END
'''
insert_rule('release funds', 'speed_check', release_funds, 'HIGH', True)

## Dynamic Application of Rules 

In [0]:
df = spark.sql("SELECT * FROM claim_policy_accident")
display(df.limit(10))

claim_no,CHASSIS_NO,policy_no,claim_date,months_as_customer,number_of_witnesses,suspicious_activity,claim_amount_injury,claim_amount_property,claim_amount_total,claim_amount_vehicle,collision_number_of_vehicles_involved,collision_type,driver_age,driver_insured_relationship,driver_license_issue_date,incident_date,incident_hour,incident_severity,incident_type,CUST_ID,POLICYTYPE,pol_issue_date,pol_eff_date,pol_expiry_date,BODY,MAKE,MODEL,MODEL_YEAR,USE_OF_VEHICLE,DRV_DOB,BOROUGH,NEIGHBORHOOD,ZIP_CODE,PRODUCT,SUM_INSURED,premium,DEDUCTABLE,ZIPCODE,address,lat_long,telematics_speed,telematics_latitude,telematics_longitude,image_name,path,modificationTime,length,content,damage_prediction,image_id,_rescued_data
0d67469e-127e-4704-a747-f3f8ed1aead7,KMHS281D1KU178638,102141775,2020-03-05,247,0,false,5300,10600,63600,47700,4,Side Collision,22.0,own-child,2018-08-28,2020-02-29,4,Trivial Damage,Multi-vehicle Collision,5568.0,COMP,2020-02-24,2020-02-24,2021-03-23,4 WD,HYUNDAI,SANTA-FE,2019.0,PRIVATE,28-02-2011,BRONX,NORTHEAST BRONX,10475.0,M 2019,80000.0,1990.0,1000,null,"BRONX, 10475.0","List(-49.27744, -153.11403)",31.59356239810586,40.784349025,-73.90740773333337,2_High.jpg,dbfs:/Volumes/main/dbdemos_fsi_smart_claims/volume_claims/Accidents/images/2_High.jpg,2025-10-23T22:19:55Z,1928661,"List(iVBORw0KGgoAAAANSUhEUgAABAAAAAQACAIAAADwf7zUAAAAemVYSWZNTQAqAAAACAACknwAAgAAAD0AAAAmkoYAAgAAABYAAABkAAAAAE9wZW5BSS0tSW1hZ2VHZW5lcmF0aW9uLS1nZW5lcmF0aW9uLVh0SkVrOHhpem5mZHE= (truncated), ...)","List(major, 0.47762542963027954)",4,null
90035736-7748-4088-8a75-8241b2cf3562,KMHJT81E2FU010417,102126329,2019-01-30,217,0,false,17680,17680,114920,79560,4,Rear Collision,43.0,other-relative,2011-01-23,2019-02-22,5,Total Loss,Multi-vehicle Collision,7957.0,COMP,2019-01-20,2019-01-20,2020-02-19,SUV,HYUNDAI,TUCSON,2015.0,PRIVATE,14-11-1981,MANHATTAN,INWOOD AND WASHINGTON HEIGHTS,10032.0,NOT CLASSIFIED,33478.0,1680.0,1000,null,"MANHATTAN, 10032.0","List(18.429148, -52.56522)",32.14801159550746,40.80451471666667,-73.92542468333335,2_High.jpg,dbfs:/Volumes/main/dbdemos_fsi_smart_claims/volume_claims/Accidents/images/2_High.jpg,2025-10-23T22:19:55Z,1928661,"List(iVBORw0KGgoAAAANSUhEUgAABAAAAAQACAIAAADwf7zUAAAAemVYSWZNTQAqAAAACAACknwAAgAAAD0AAAAmkoYAAgAAABYAAABkAAAAAE9wZW5BSS0tSW1hZ2VHZW5lcmF0aW9uLS1nZW5lcmF0aW9uLVh0SkVrOHhpem5mZHE= (truncated), ...)","List(major, 0.47762542963027954)",4,null
f5c3d29f-a0c8-48f3-96ba-4522fdb14c5f,6G1EK52B87L866741,102076391,2016-03-16,212,1,false,5810,11620,63910,46480,1,null,33.0,unmarried,2005-08-15,2016-03-03,15,Major Damage,Parked Car,8045.0,COMP,2016-03-06,2016-03-06,2017-04-05,SALOON,CHEVROLET,LUMINA,2007.0,PRIVATE,28-06-2006,MANHATTAN,LOWER MANHATTAN,10280.0,STANDARD,20000.0,1090.0,1000,null,"MANHATTAN, 10280.0","List(58.648987, -13.586999)",32.14801159550746,40.80451471666667,-73.92542468333335,2_High.jpg,dbfs:/Volumes/main/dbdemos_fsi_smart_claims/volume_claims/Accidents/images/2_High.jpg,2025-10-23T22:19:55Z,1928661,"List(iVBORw0KGgoAAAANSUhEUgAABAAAAAQACAIAAADwf7zUAAAAemVYSWZNTQAqAAAACAACknwAAgAAAD0AAAAmkoYAAgAAABYAAABkAAAAAE9wZW5BSS0tSW1hZ2VHZW5lcmF0aW9uLS1nZW5lcmF0aW9uLVh0SkVrOHhpem5mZHE= (truncated), ...)","List(major, 0.47762542963027954)",4,null
6fee8211-bc67-4bd8-b5a6-6322418e2b50,LB3774KZ1LH003062,102146520,2020-07-05,11,3,false,2200,4400,24200,17600,1,null,29.0,unmarried,2016-10-13,2020-07-02,0,Trivial Damage,Single Vehicle Collision,7620.0,COMP,2020-06-23,2020-06-25,2021-07-24,SUV,GEELY,EMGRAND GS SP,2020.0,PRIVATE,24-04-2007,BRONX,HUNTS POINT AND MOTT HAVEN,10474.0,M 2019,56000.0,1970.0,2000,null,"BRONX, 10474.0","List(-17.101765, -3.675996)",32.14801159550746,40.80451471666667,-73.92542468333335,2_High.jpg,dbfs:/Volumes/main/dbdemos_fsi_smart_claims/volume_claims/Accidents/images/2_High.jpg,2025-10-23T22:19:55Z,1928661,"List(iVBORw0KGgoAAAANSUhEUgAABAAAAAQACAIAAADwf7zUAAAAemVYSWZNTQAqAAAACAACknwAAgAAAD0AAAAmkoYAAgAAABYAAABkAAAAAE9wZW5BSS0tSW1hZ2VHZW5lcmF0aW9uLS1nZW5lcmF0aW9uLVh0SkVrOHhpem5mZHE= (truncated), ...)

In [0]:
rules = spark.sql('SELECT * FROM claims_rules where is_active=true order by rule_id').collect()
for rule in rules:
  print(rule.rule, rule.check_code)
  df=df.withColumn(rule.check_name, F.expr(rule.check_code))

#overwrite table with new insights
df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("claim_insights")

invalid policy date 
  CASE WHEN to_date(pol_eff_date, "yyyy-MM-dd") < to_date(claim_date) and to_date(pol_expiry_date, "yyyy-MM-dd") < to_date(claim_date) THEN "VALID" 
  ELSE "NOT VALID"  
  END

exceeds policy amount 
CASE WHEN  sum_insured >= claim_amount_total 
    THEN "calim value in the range of premium"
    ELSE "claim value more than premium"
END 

severity mismatch 
CASE WHEN    damage_prediction.label="major" AND damage_prediction.score > 0.8 THEN  "Severity matches the report"
       WHEN  damage_prediction.label="minor" AND damage_prediction.score > 0.6 THEN  "Severity matches the report"
       WHEN  damage_prediction.label="ok" AND damage_prediction.score > 0.4 THEN  "Severity matches the report"
       ELSE "Severity does not match"
END 

exceeds speed 
CASE WHEN  telematics_speed <= 45 and telematics_speed > 0 THEN  "Normal Speed"
       WHEN telematics_speed > 45 THEN  "High Speed"
       ELSE "Invalid speed"
END

release funds 
CASE WHEN  reported_severity_check="Se

In [0]:
%sql
SELECT valid_date, valid_amount, reported_severity_check FROM claim_insights

valid_date,valid_amount,reported_severity_check
NOT VALID,calim value in the range of premium,Severity does not match
NOT VALID,claim value more than premium,Severity does not match
NOT VALID,claim value more than premium,Severity does not match
NOT VALID,calim value in the range of premium,Severity does not match
NOT VALID,claim value more than premium,Severity does not match
NOT VALID,calim value in the range of premium,Severity does not match
NOT VALID,claim value more than premium,Severity does not match
NOT VALID,claim value more than premium,Severity does not match
NOT VALID,calim value in the range of premium,Severity does not match
NOT VALID,claim value more than premium,Severity does not match



Some data points can be checked out easily using <b>simple rules</b>, others may require <b>ML models </b>to score the data to produce the desired insights. In this notebook and the previous one, we have demonstrated how the insights generated from both can be consolidated into <b> structured </b> tabular data for easy consumption from a <b> dashboard </b> by various stakeholders including executives and investigation officers who use the same data but for completely different purposes. <br>
Moreover, all these insights are well-secured by <b> Unity Catalog </b>. So if the data, model, or insight is sensitive and is meant for select audiences then using a few simple 'GRANT' statements can ensure that it is adequately protected and is never accidentally exposed to the wrong party. Dynamic masking can be used to hide PII data. In the next notebook, we'll see how we can put all of this on auto-drive using Databricks workflows.